# 13 - Finale Hard-Set Batch Commands

Dieses Notebook sammelt die 210 single-site Hard-Tasks aus WebArena Verified, vergleicht sie mit dem bestehenden Hauptlauf und erzeugt Terminal-Befehle in 5er-Batches.

Ziel: alle Ergebnisse landen weiter in demselben Experimentordner `hk-agent-browsergym-planact-main-v02_basis_v3`, ohne dass bestehende Eintraege geloescht werden. Mit `--resume-summary` werden bereits vorhandene `(task_id, h, k)`-Kombinationen uebersprungen.

In [1]:

from pathlib import Path
import json
import math
import textwrap

import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

EXPERIMENT_NAME = "hk-agent-browsergym-planact-main-v02_basis_v3"
RUN_ROOT = ROOT / "runs" / "hk-agent" / EXPERIMENT_NAME
SUMMARY_CSV = RUN_ROOT / "summary.csv"
SUMMARY_JSON = RUN_ROOT / "summary.json"

DATASET_PATH = ROOT / "external" / "webarena-verified" / "assets" / "dataset" / "webarena-verified.json"
HARD_SUBSET_PATH = ROOT / "external" / "webarena-verified" / "assets" / "dataset" / "subsets" / "webarena-verified-hard.json"

SUPPORTED_SINGLE_SITES = {"gitlab", "reddit", "shopping", "shopping_admin"}

# Finale H/k-Matrix fuer den Hauptlauf. Fuer kleine Retests temporaer reduzieren,
# z.B. HS = [2], KS = [10].
HS = [0, 2, 5, 10]
KS = [0, 2, 5, 10]

# 5er-Schritte sind langsamer als 10er-Batches, aber besser kontrollierbar
# und bei GitLab/Mutation-Runs leichter wiederaufnehmbar.
BATCH_SIZE = 5

# Schaetzung nur fuer Planung/Anzeige, nicht fuer den Befehl.
ASSUMED_MINUTES_PER_RUN = 10

BASE_COMMAND_OPTIONS = {
    "run_mode": "agent",
    "planner_model": "gemma4:26b",
    "executor_model": "gemma4:e4b",
    "agent_architecture": "v3",
    "max_steps_policy": "tiered",
    "max_steps_navigation": 20,
    "max_steps_retrieval": 30,
    "max_steps_policy_task": 25,
    "max_steps_mutation": 50,
    # 0 deaktiviert das harte Planner-Call-Limit. Planner-Calls bleiben als
    # Kosten-/Laufzeitvariable in summary.csv erhalten, brechen den Run aber
    # nicht mehr vor dem Step-Budget ab.
    "max_planner_calls": 0,
    "planner_call_margin": 2,
    "max_steps": 500,
    "llm_timeout_seconds": 600,
    "max_consecutive_llm_timeouts": 0,
    "success_policy": "contamination_adjusted",
    "ollama_base_url": "http://127.0.0.1:11435",
    "reset_site_before_mutate": True,
    "site_reset_timeout_seconds": 300,
}

print("ROOT:", ROOT)
print("RUN_ROOT:", RUN_ROOT)
print("SUMMARY_CSV exists:", SUMMARY_CSV.exists())


ROOT: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code
RUN_ROOT: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/runs/hk-agent/hk-agent-browsergym-planact-main-v02_basis_v3
SUMMARY_CSV exists: True


In [2]:
def extract_task_type(task: dict) -> str:
    """Use the official evaluator metadata as the source of truth."""
    values = []
    for evaluator in task.get("eval", []) or []:
        expected = evaluator.get("expected", {}) or {}
        if expected.get("task_type"):
            values.append(str(expected["task_type"]).upper())
    if values:
        return values[0]
    intent = str(task.get("intent", "")).lower()
    if any(word in intent for word in ["change", "create", "delete", "add", "remove", "update", "set", "vote", "mark", "invite"]):
        return "MUTATE"
    return "RETRIEVE"

all_tasks = json.loads(DATASET_PATH.read_text())
hard_subset = json.loads(HARD_SUBSET_PATH.read_text())
hard_ids = {int(task_id) for task_id in hard_subset["task_ids"]}

rows = []
for task in all_tasks:
    task_id = int(task["task_id"])
    sites = [str(site) for site in task.get("sites", [])]
    if task_id not in hard_ids:
        continue
    if len(sites) != 1:
        continue
    site = sites[0]
    if site not in SUPPORTED_SINGLE_SITES:
        continue
    rows.append({
        "task_id": task_id,
        "site": site,
        "task_type": extract_task_type(task),
        "intent_template_id": task.get("intent_template_id"),
        "revision": task.get("revision"),
        "gym_id": f"browsergym/webarena_verified.{task_id}",
        "intent": task.get("intent", ""),
    })

hard_single = pd.DataFrame(rows).sort_values(["site", "task_type", "task_id"]).reset_index(drop=True)
assert len(hard_single) == 210, f"Expected 210 single-site hard tasks, got {len(hard_single)}"

print("Hard single-site tasks:", len(hard_single))
display(pd.crosstab(hard_single["site"], hard_single["task_type"], margins=True))

Hard single-site tasks: 210


task_type,MUTATE,NAVIGATE,RETRIEVE,All
site,,,,
gitlab,36,6,15,57
reddit,36,0,6,42
shopping,21,10,25,56
shopping_admin,26,6,23,55
All,119,22,69,210


In [3]:

expected_hk_pairs = {(h, k) for h in HS for k in KS}
expected_runs_per_task = len(expected_hk_pairs)

if SUMMARY_CSV.exists():
    summary = pd.read_csv(SUMMARY_CSV)
else:
    summary = pd.DataFrame()

if not summary.empty:
    summary = summary.copy()
    summary["task_id"] = summary["task_id"].astype(int)
    summary["h"] = summary["h"].astype(int)
    summary["k"] = summary["k"].astype(int)
    # Ein H/k-Paar zaehlt nur als erledigt, wenn der Agentenlauf wirklich
    # completed ist und official_success nicht leer/NaN ist. Login-/Docker-
    # Runtime-Exceptions werden dadurch erneut in die Missing-Liste aufgenommen.
    valid_run_mask = (summary["status"].fillna("") == "completed") & summary["official_success"].notna()
    valid_summary = summary[valid_run_mask].copy()
    summary_task_ids = set(summary["task_id"].dropna().astype(int))
    observed_pairs_by_task = (
        valid_summary.groupby("task_id")[["h", "k"]]
        .apply(lambda x: set(map(tuple, x[["h", "k"]].to_numpy())))
        .to_dict()
    )
    invalid_rows = summary[~valid_run_mask].copy()
else:
    summary_task_ids = set()
    observed_pairs_by_task = {}
    invalid_rows = pd.DataFrame()

hard_ids_single = set(hard_single["task_id"].astype(int))
complete_task_ids = {
    task_id
    for task_id in hard_ids_single
    if expected_hk_pairs.issubset(observed_pairs_by_task.get(task_id, set()))
}
partial_task_ids = {
    task_id
    for task_id in hard_ids_single
    if task_id in observed_pairs_by_task and task_id not in complete_task_ids
}
missing_task_ids = sorted(hard_ids_single - complete_task_ids)
new_task_ids = sorted(hard_ids_single - summary_task_ids)

status_rows = []
for task_id in sorted(hard_ids_single):
    observed = observed_pairs_by_task.get(task_id, set())
    missing_pairs = sorted(expected_hk_pairs - observed)
    invalid_for_task = invalid_rows[invalid_rows["task_id"] == task_id] if not invalid_rows.empty else pd.DataFrame()
    status_rows.append({
        "task_id": task_id,
        "done_pairs": len(observed & expected_hk_pairs),
        "expected_pairs": expected_runs_per_task,
        "missing_pairs": len(missing_pairs),
        "invalid_pairs": len(invalid_for_task),
        "complete": len(missing_pairs) == 0,
    })
status_df = hard_single.merge(pd.DataFrame(status_rows), on="task_id", how="left")

print("Summary rows:", len(summary))
print("Invalid/non-completed rows:", len(invalid_rows))
print("Hard single-site complete task ids:", len(complete_task_ids))
print("Hard single-site partial task ids:", len(partial_task_ids))
print("Hard single-site incomplete/missing task ids:", len(missing_task_ids))
print("Completely new task ids:", len(new_task_ids))

display(status_df.groupby(["site", "task_type", "complete"], dropna=False).agg(
    tasks=("task_id", "count"),
    missing_hk_pairs=("missing_pairs", "sum"),
    invalid_hk_pairs=("invalid_pairs", "sum"),
).reset_index().sort_values(["site", "task_type", "complete"]))

display(status_df[~status_df["complete"]][["task_id", "site", "task_type", "done_pairs", "missing_pairs", "invalid_pairs", "intent"]].head(30))

if not invalid_rows.empty:
    display(invalid_rows[["task_id", "h", "k", "status", "failure_category", "error", "output_dir"]].head(20))


Summary rows: 2193
Invalid/non-completed rows: 42
Hard single-site complete task ids: 128
Hard single-site partial task ids: 10
Hard single-site incomplete/missing task ids: 82
Completely new task ids: 71


,site,task_type,complete,tasks,missing_hk_pairs,invalid_hk_pairs
0,gitlab,MUTATE,False,1,6,6
1,gitlab,MUTATE,True,35,0,0
2,gitlab,NAVIGATE,True,6,0,0
3,gitlab,RETRIEVE,True,15,0,0
4,reddit,MUTATE,False,3,15,5
5,reddit,MUTATE,True,33,0,0
6,reddit,RETRIEVE,False,1,12,0
7,reddit,RETRIEVE,True,5,0,0
8,shopping,MUTATE,False,5,38,29
9,shopping,MUTATE,True,16,0,0


,task_id,site,task_type,done_pairs,missing_pairs,invalid_pairs,intent
35,810,gitlab,MUTATE,10,6,6,Assign the issue regarding flash alert bug in ...
87,721,reddit,MUTATE,12,4,4,Like all submissions created by UniversityofBa...
88,724,reddit,MUTATE,15,1,1,Like all submissions created by Hrekires in fo...
89,730,reddit,MUTATE,6,10,0,DisLike all submissions created by Hrekires in...
97,67,reddit,RETRIEVE,4,12,0,"Among the top 10 hottest posts in the ""Books"" ..."
99,431,shopping,MUTATE,14,2,2,Add the product with the lowest per unit price...
101,435,shopping,MUTATE,6,10,10,Add the product with the lowest per unit price...
106,519,shopping,MUTATE,0,16,16,Add the product on the current page to my wish...
111,530,shopping,MUTATE,7,9,0,Fill out the contact us form with this refund ...
112,571,shopping,MUTATE,15,1,1,"I recently moved, my address is 231 Willow Way..."


,task_id,h,k,status,failure_category,error,output_dir
1498,810,5,5,skipped_existing,step_budget_exhausted,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
1499,810,5,10,skipped_existing,step_budget_exhausted,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
1500,810,10,0,skipped_existing,official_eval_mismatch,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
1501,810,10,2,skipped_existing,missing_required_mutation,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
1502,810,10,5,skipped_existing,missing_required_mutation,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
1503,810,10,10,skipped_existing,step_budget_exhausted,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
1521,721,0,2,skipped_existing,step_budget_exhausted,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
1522,721,0,5,skipped_existing,step_budget_exhausted,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
1523,721,0,10,skipped_existing,step_budget_exhausted,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
1524,721,2,0,skipped_existing,step_budget_exhausted,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...


In [4]:

def chunked(items, n):
    return [items[i:i+n] for i in range(0, len(items), n)]

def shell_join_values(values):
    return " ".join(str(v) for v in values)

def make_command(task_ids):
    lines = [
        "NODE_OPTIONS=--max-old-space-size=8192 python scripts/run_hk_agent_experiment.py \\",
        f"  --experiment-name {EXPERIMENT_NAME} \\",
        f"  --task-ids {shell_join_values(task_ids)} \\",
        f"  --hs {shell_join_values(HS)} \\",
        f"  --ks {shell_join_values(KS)} \\",
        f"  --run-mode {BASE_COMMAND_OPTIONS['run_mode']} \\",
        f"  --planner-model {BASE_COMMAND_OPTIONS['planner_model']} \\",
        f"  --executor-model {BASE_COMMAND_OPTIONS['executor_model']} \\",
        f"  --agent-architecture {BASE_COMMAND_OPTIONS['agent_architecture']} \\",
        f"  --max-steps-policy {BASE_COMMAND_OPTIONS['max_steps_policy']} \\",
        f"  --max-steps-navigation {BASE_COMMAND_OPTIONS['max_steps_navigation']} \\",
        f"  --max-steps-retrieval {BASE_COMMAND_OPTIONS['max_steps_retrieval']} \\",
        f"  --max-steps-policy-task {BASE_COMMAND_OPTIONS['max_steps_policy_task']} \\",
        f"  --max-steps-mutation {BASE_COMMAND_OPTIONS['max_steps_mutation']} \\",
        f"  --max-planner-calls {BASE_COMMAND_OPTIONS['max_planner_calls']} \\",
        f"  --planner-call-margin {BASE_COMMAND_OPTIONS['planner_call_margin']} \\",
        f"  --max-steps {BASE_COMMAND_OPTIONS['max_steps']} \\",
        f"  --llm-timeout-seconds {BASE_COMMAND_OPTIONS['llm_timeout_seconds']} \\",
        f"  --max-consecutive-llm-timeouts {BASE_COMMAND_OPTIONS['max_consecutive_llm_timeouts']} \\",
        f"  --success-policy {BASE_COMMAND_OPTIONS['success_policy']} \\",
        f"  --ollama-base-url {BASE_COMMAND_OPTIONS['ollama_base_url']} \\",
        "  --reset-site-before-mutate \\",
        f"  --site-reset-timeout-seconds {BASE_COMMAND_OPTIONS['site_reset_timeout_seconds']} \\",
        "  --resume-summary \\",
        "  --refresh-existing-diagnostics",
    ]
    return "\n".join(lines)

# Sortierung: erst Site, dann Typ, dann ID. Dadurch bleiben Batches fachlich besser interpretierbar.
missing_ordered = (
    status_df[~status_df["complete"]]
    .sort_values(["site", "task_type", "task_id"])["task_id"]
    .astype(int)
    .tolist()
)

batches = chunked(missing_ordered, BATCH_SIZE)
commands = [make_command(batch) for batch in batches]

batch_rows = []
for idx, task_ids in enumerate(batches, start=1):
    subset = hard_single[hard_single["task_id"].isin(task_ids)]
    n_runs_total = len(task_ids) * expected_runs_per_task
    n_runs_remaining = int(status_df[status_df["task_id"].isin(task_ids)]["missing_pairs"].sum())
    n_invalid = int(status_df[status_df["task_id"].isin(task_ids)]["invalid_pairs"].sum())
    batch_rows.append({
        "batch": idx,
        "task_count": len(task_ids),
        "remaining_hk_runs": n_runs_remaining,
        "invalid_hk_runs": n_invalid,
        "max_hk_runs_if_all_new": n_runs_total,
        "estimated_hours_at_10min": round(n_runs_remaining * ASSUMED_MINUTES_PER_RUN / 60, 1),
        "sites": ", ".join(sorted(subset["site"].unique())),
        "task_types": ", ".join(sorted(subset["task_type"].unique())),
        "task_ids": " ".join(map(str, task_ids)),
    })

batch_df = pd.DataFrame(batch_rows)
print("Batches:", len(batch_df))
display(batch_df)


Batches: 17


,batch,task_count,remaining_hk_runs,invalid_hk_runs,max_hk_runs_if_all_new,estimated_hours_at_10min,sites,task_types,task_ids
0,1,5,33,11,80,5.5,"gitlab, reddit","MUTATE, RETRIEVE",810 721 724 730 67
1,2,5,38,29,80,6.3,shopping,MUTATE,431 435 519 530 571
2,3,5,66,2,80,11.0,shopping,"NAVIGATE, RETRIEVE",273 124 125 142 143
3,4,5,80,0,80,13.3,shopping,RETRIEVE,147 148 149 163 165
4,5,5,80,0,80,13.3,shopping,RETRIEVE,166 191 226 235 320
5,6,5,80,0,80,13.3,shopping,RETRIEVE,321 323 335 337 338
6,7,5,80,0,80,13.3,"shopping, shopping_admin","MUTATE, RETRIEVE",388 488 489 491 493
7,8,5,80,0,80,13.3,shopping_admin,MUTATE,499 502 544 545 546
8,9,5,80,0,80,13.3,shopping_admin,MUTATE,549 550 551 694 696
9,10,5,80,0,80,13.3,shopping_admin,MUTATE,697 701 702 703 769


In [5]:

COMMANDS_MD = RUN_ROOT / "hardset_missing_batch_commands.md"
COMMANDS_SH = RUN_ROOT / "hardset_missing_batch_commands.sh"

md_parts = [
    f"# Missing hard-set batch commands for `{EXPERIMENT_NAME}`\n",
    f"Generated from `{Path('notebooks/13_final_hardset_batch_commands.ipynb')}`.\n",
    f"Hard single-site tasks: {len(hard_single)}\n",
    f"Complete task ids: {len(complete_task_ids)}\n",
    f"Incomplete/missing task ids: {len(missing_ordered)}\n",
    f"H values: {HS}\n",
    f"K values: {KS}\n",
    "\nHinweis: Alle Befehle schreiben in denselben Experimentordner. `--resume-summary` ueberspringt bereits vorhandene gueltige `(task_id, h, k)`-Kombinationen. Runtime-/Login-Fehler ohne gueltigen Agentenlauf werden erneut aufgenommen. MUTATE-Runs starten mit `--reset-site-before-mutate`, damit stateful Tasks sauberer vergleichbar bleiben.\n",
]

sh_parts = ["#!/usr/bin/env bash", "set -euo pipefail", ""]
for idx, command in enumerate(commands, start=1):
    md_parts.append(f"\n## Batch {idx}\n")
    md_parts.append("```bash\n" + command + "\n```\n")
    sh_parts.append(f"# Batch {idx}")
    sh_parts.append(command)
    sh_parts.append("")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
COMMANDS_MD.write_text("\n".join(md_parts))
COMMANDS_SH.write_text("\n".join(sh_parts))

print("Wrote:", COMMANDS_MD)
print("Wrote:", COMMANDS_SH)

for idx, command in enumerate(commands, start=1):
    print(f"\n# Batch {idx}/{len(commands)}")
    print(command)


Wrote: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/runs/hk-agent/hk-agent-browsergym-planact-main-v02_basis_v3/hardset_missing_batch_commands.md
Wrote: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/runs/hk-agent/hk-agent-browsergym-planact-main-v02_basis_v3/hardset_missing_batch_commands.sh

# Batch 1/17
NODE_OPTIONS=--max-old-space-size=8192 python scripts/run_hk_agent_experiment.py \
  --experiment-name hk-agent-browsergym-planact-main-v02_basis_v3 \
  --task-ids 810 721 724 730 67 \
  --hs 0 2 5 10 \
  --ks 0 2 5 10 \
  --run-mode agent \
  --planner-model gemma4:26b \
  --executor-model gemma4:e4b \
  --agent-architecture v3 \
  --max-steps-policy tiered \
  --max-steps-navigation 20 \
  --max-steps-retrieval 30 \
  --max-steps-policy-task 25 \
  --max-steps-mutation 50 \
  --max-planner-calls 0 \
  --planner-call-margin 2 \
  --max-steps 500 \
  --llm-timeout-seconds 600 \
  --max-consecutive-llm-timeouts 0 \
  --success-policy contamina


## Wichtig fuer die Interpretation

- Ein Task gilt hier erst als vollstaendig, wenn alle `h x k`-Kombinationen aus `HS` und `KS` als gueltige `completed`-Runs in `summary.csv` vorhanden sind.
- Runtime-/Login-Fehler, bei denen der Agent gar nicht sauber gestartet ist, werden nicht als Agentenleistung gewertet und bleiben in der Missing-Liste.
- `NODE_OPTIONS=--max-old-space-size=8192` ist gesetzt, weil lange Playwright-Laeufe vorher an einem Node-Heap-OOM gescheitert sind.
- Die Befehle nutzen bewusst `python ...` statt `uv run python ...`, damit die aktive Python-3.12-`.venv` nicht durch `uv`/Projektauflösung ersetzt wird.
- `--max-planner-calls 0` deaktiviert das harte Planner-Call-Limit. Planner-Aufrufe werden weiter geloggt und gehen in Token-/Zeitkosten ein, brechen den Run aber nicht künstlich vor dem Step-Budget ab.
- `--max-steps-mutation 50` ist der aktuelle Fail-fast-Kompromiss: echte GitLab-/Formular-Loops werden nicht unnötig bis 80/150 Schritte verlängert.
- Wenn du nur testen willst, setze oben temporaer `HS = [2]`, `KS = [2]` oder `BATCH_SIZE = 1`.



## Diagnose-Refresh ohne neue Runs

Wenn alle fehlenden Runs abgeschlossen sind, kannst du die Near-Miss- und Failure-Class-Spalten aus den vorhandenen Artefakten neu berechnen lassen, ohne BrowserGym erneut auszuführen:

```bash
python scripts/run_hk_agent_experiment.py \
  --experiment-name hk-agent-browsergym-planact-main-v02_basis_v3 \
  --task-ids <TASK_IDS> \
  --hs 0 2 5 10 \
  --ks 0 2 5 10 \
  --run-mode agent \
  --planner-model gemma4:26b \
  --executor-model gemma4:e4b \
  --agent-architecture v3 \
  --success-policy contamination_adjusted \
  --resume-summary \
  --refresh-existing-only \
  --refresh-existing-diagnostics
```

Fuer `<TASK_IDS>` kannst du die final untersuchten IDs einsetzen oder eine kleinere Gruppe, wenn du nur einen Teil aktualisieren willst.



## Praktischer Ablauf

1. Notebook ausfuehren und `hardset_missing_batch_commands.md`/`.sh` erzeugen.
2. Batches nacheinander im Terminal starten.
3. Bei GitLab-Login-Runtime-Exceptions denselben `(task_id, h, k)`-Run ersetzen; echte Agent-Loops bleiben als Failure-Klasse stehen.
4. Am Ende Diagnose-Refresh ausfuehren und danach im Analyse-Notebook auswerten.
